# NBIM Holdings – Setup & Load
Fetches annual holdings data from NBIM's public API (1998–present),
cleans strings, writes TSV files, and bulk-loads into MySQL.

In [13]:
import os
import requests
import pandas as pd
import numpy as np
import mysql.connector
from dotenv import load_dotenv
import os

In [10]:
# --- Config ---
YEARS   = list(range(1998, 2026))
BASE    = "https://www.nbim.no/api/investments/v2"
V       = "9f492697"
HEADERS = {"User-Agent": "Mozilla/5.0"}
CSV_DIR = r".\nbim_csv"

load_dotenv()

DB_CONFIG = dict(
    host               = "localhost",
    user               = "root",
    password           = os.environ.get("NBIM_DB_PASSWORD"),
    database           = "nbim",
    allow_local_infile = True,
)

os.makedirs(CSV_DIR, exist_ok=True)

In [3]:
# --- Helper functions ---

STR_COLS = ["company", "sector", "asset_class", "country_code", "incorporated"]

def clean_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace and normalise case on all string columns."""
    for col in STR_COLS:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()
    return df


def build_df_for_year(year: int) -> pd.DataFrame:
    url = f"{BASE}/{year}-12-31.json?v={V}"
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()

    df = pd.json_normalize(data["data"])

    keep = ["id", "n", "cc", "a.e", "a.n", "o", "eq.vid", "eq.s", "eq.icc", "fi.s", "fi.icc", "at"]
    df = df[[c for c in keep if c in df.columns]]

    eq = df[df["at"] == 0].copy()
    eq["asset_class"] = "Equity"

    fi = df[df["at"] == 1].copy()
    fi["asset_class"] = "Fixed Income"

    out = pd.concat([eq, fi], ignore_index=True).rename(columns={
        "id":    "nbim_id",
        "n":     "company",
        "cc":    "country_code",
        "a.e":   "value_usd",
        "o":     "ownership",
        "eq.vid": "ticker",
    })

    out["sector"]      = out["eq.s"].combine_first(out["fi.s"])
    out["incorporated"] = out["eq.icc"].combine_first(out["fi.icc"])
    out["year"]         = year

    out = out[[
        "year", "asset_class", "nbim_id", "company", "ticker",
        "sector", "ownership", "value_usd", "country_code", "incorporated"
    ]]

    out = clean_strings(out)
    return out.replace({np.nan: None})


def write_csv(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False, sep="\t", na_rep="\\N", lineterminator="\n")


def load_csv_mysql(conn, csv_path: str, table="nbim_holdings"):
    cur = conn.cursor()
    sql = f"""
    LOAD DATA LOCAL INFILE %s
    INTO TABLE {table}
    CHARACTER SET utf8mb4
    FIELDS TERMINATED BY '\\t'
    LINES TERMINATED BY '\\n'
    IGNORE 1 LINES
    (year, asset_class, nbim_id, company, ticker, sector,
     ownership, value_usd, country_code, incorporated)
    """
    cur.execute(sql, (csv_path,))
    conn.commit()
    cur.close()


def ensure_conn(conn):
    try:
        if conn is None or not conn.is_connected():
            raise Exception("reconnect")
        return conn
    except Exception:
        return mysql.connector.connect(**DB_CONFIG)

In [4]:
# --- Fetch, clean, and load ---
conn   = mysql.connector.connect(**DB_CONFIG)
failed = []

for year in YEARS:
    conn = ensure_conn(conn)
    try:
        df_year  = build_df_for_year(year)
        csv_path = os.path.join(CSV_DIR, f"nbim_{year}.tsv")
        write_csv(df_year, csv_path)
        load_csv_mysql(conn, csv_path)
        print(f"{year}: loaded {len(df_year):,} rows")
    except Exception as e:
        failed.append((year, str(e)))
        print(f"{year}: FAILED -> {e}")

conn.close()

if failed:
    print(f"\n{len(failed)} year(s) failed:")
    for year, err in failed:
        print(f"  {year}: {err}")
else:
    print("\nAll years loaded successfully.")

1998: loaded 2,150 rows
1999: loaded 2,087 rows
2000: loaded 1,840 rows
2001: loaded 2,389 rows
2002: loaded 2,942 rows
2003: loaded 3,601 rows
2004: loaded 4,143 rows
2005: loaded 4,613 rows
2006: loaded 4,945 rows
2007: loaded 8,994 rows
2008: loaded 10,087 rows
2009: loaded 10,289 rows
2010: loaded 10,181 rows
2011: loaded 9,405 rows
2012: loaded 8,624 rows
2013: loaded 9,273 rows
2014: loaded 10,277 rows
2015: loaded 10,328 rows
2016: loaded 10,235 rows
2017: loaded 10,368 rows
2018: loaded 10,411 rows
2019: loaded 10,379 rows
2020: loaded 10,368 rows
2021: loaded 10,703 rows
2022: loaded 10,658 rows
2023: loaded 10,248 rows
2024: loaded 10,166 rows
2025: loaded 8,819 rows

All years loaded successfully.


In [6]:
# --- Verify row counts ---
conn = mysql.connector.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("SELECT COUNT(*) FROM nbim_holdings")
print(f"Total rows: {cur.fetchone()[0]:,}")

print("\nRows per year:")
cur.execute("""
    SELECT year, COUNT(*) AS row_count
    FROM nbim_holdings
    GROUP BY year
    ORDER BY year
""")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]:,}")

cur.close()
conn.close()

Total rows: 218,523

Rows per year:
  1998: 2,150
  1999: 2,087
  2000: 1,840
  2001: 2,389
  2002: 2,942
  2003: 3,601
  2004: 4,143
  2005: 4,613
  2006: 4,945
  2007: 8,994
  2008: 10,087
  2009: 10,289
  2010: 10,181
  2011: 9,405
  2012: 8,624
  2013: 9,273
  2014: 10,277
  2015: 10,328
  2016: 10,235
  2017: 10,368
  2018: 10,411
  2019: 10,379
  2020: 10,368
  2021: 10,703
  2022: 10,658
  2023: 10,248
  2024: 10,166
  2025: 8,819
